In [1]:
import itertools
import sys
import numpy as np
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

2025-12-24 09:47:35.657470: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-24 09:47:35.668299: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-24 09:47:35.680651: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-24 09:47:35.684637: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-24 09:47:35.693883: I tensorflow/core/platform/cpu_feature_guar

In [2]:
X_tr_np, X_val_np, X_train_np, X_test_np, y_tr_np, y_val_np, y_train_np, y_test_np = load(norm='tanh_norm')
norm_options = ['norm', 'tanh', 'tanh_norm']
hidden_options = [
    [8192, 8192], [4096, 4096], [2048, 2048],
    [8192, 4096], [4096, 2048], [4096, 4096, 4096],
    [2048, 2048, 2048], [4096, 2048, 1024], [8192, 4096, 2048]]
lr_options = [1e-2, 1e-3, 1e-4, 1e-5]
dropout_options = [(0, 0), (0.2, 0.5)]
hyperparameter_grid = list(itertools.product(
    norm_options, hidden_options, lr_options, dropout_options))
best_val_loss = np.inf
best_params = None
best_epoch = None

In [3]:
def moving_average(x, n):
    x = np.asarray(x)
    return np.convolve(x, np.ones(n)/n, mode='valid')

In [ ]:
for norm_type, hidden_layers, lr, (input_do, hidden_do) in hyperparameter_grid:
    X_tr_norm = X_tr_np.copy()
    X_val_norm = X_val_np.copy()
    model = Sequential()
    for i, units in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(units, input_shape=(X_tr_norm.shape[1],),
                            activation='relu', kernel_initializer='he_normal'))
            if input_do > 0:
                model.add(Dropout(input_do))
        elif i == len(hidden_layers) - 1:
            model.add(Dense(units, activation='linear', kernel_initializer='he_normal'))
        else:
            model.add(Dense(units, activation='relu', kernel_initializer='he_normal'))
            if hidden_do > 0:
                model.add(Dropout(hidden_do))
    model.add(Dense(1, activation='linear', kernel_initializer='he_normal'))
    model.compile(loss='mean_squared_error', optimizer=SGD(learning_rate=lr, momentum=0.5))
    hist = model.fit(
        X_tr_norm, y_tr_np,
        validation_data=(X_val_norm, y_val_np),
        epochs=50, batch_size=64, shuffle=True, verbose=0)
    val_losses = hist.history['val_loss']
    ma_losses = moving_average(val_losses, n=25)
    best_ma_loss = np.min(ma_losses)
    best_ma_epoch = np.argmin(ma_losses) + 25
    if best_ma_loss < best_val_loss:
        best_val_loss = best_ma_loss
        best_epoch = best_ma_epoch
        best_params = {
            "norm": norm_type,
            "layers": hidden_layers,
            "learning_rate": lr,
            "input_dropout": input_do,
            "hidden_dropout": hidden_do,
            "epochs": best_ma_epoch
        }

I0000 00:00:1766569664.914940     184 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1766569664.981839     184 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1766569664.981869     184 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1766569664.983724     184 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1766569664.983742     184 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

ResourceExhaustedError: in user code:

    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py", line 1398, in train_function  *
        return step_function(self, iterator)
    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py", line 1381, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py", line 1370, in run_step  **
        outputs = model.train_step(data)
    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py", line 1151, in train_step
        self.optimizer.minimize(loss, self.trainable_variables, tape=tape)
    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/optimizers/optimizer.py", line 621, in minimize
        self.apply_gradients(grads_and_vars)
    File "/usr/local/lib/python3.12/dist-packages/tf_keras/src/optimizers/optimizer.py", line 1300, in apply_gradients
 

In [ ]:
out_file = ROOT_DIR / "best_hyperparams.txt"
with open(out_file, "w") as f:
    for k, v in best_params.items():
        f.write(f"{k}:{v}\n")
    f.write(f"best_val_loss:{best_val_loss}\n")